
# 09 — Uplift Modeling and Targeting

Notebook 06 showed that treatment improves conversion overall. This notebook asks:

> Which users are most likely to benefit from treatment?

Models:

- **T-Learner Logistic Regression**
- **S-Learner HistGradientBoosting**

The cleaned Criteo dataset is sampled reproducibly across Parquet parts so the notebook remains laptop-friendly.


In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.home() / "Desktop" / "resume_projects" / "ProductPulse"
PROCESSED_02 = PROJECT_ROOT / "data" / "processed" / "02_cleaned"
PROCESSED_07 = PROJECT_ROOT / "data" / "processed" / "07_modeling"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

PROCESSED_07.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


## 1. Build development sample

In [ ]:

CRITEO_DIR = PROCESSED_02 / "criteo_uplift"
parts = sorted(CRITEO_DIR.glob("*.parquet"))
if not parts:
    raise FileNotFoundError(f"No Criteo parquet parts found in {CRITEO_DIR}")

feature_cols = [f"f{i}" for i in range(12)]
required_cols = feature_cols + ["treatment", "conversion"]

SAMPLE_PER_PART = 25_000
RANDOM_STATE = 42

sample_frames = []
for i, part in enumerate(parts):
    batch = pd.read_parquet(part, columns=required_cols)
    n = min(SAMPLE_PER_PART, len(batch))
    sample_frames.append(
        batch.sample(n=n, random_state=RANDOM_STATE + i)
    )

criteo = pd.concat(sample_frames, ignore_index=True)

strata = (
    criteo["treatment"].astype(str)
    + "_"
    + criteo["conversion"].astype(str)
)

train_df, test_df = train_test_split(
    criteo,
    test_size=0.25,
    random_state=42,
    stratify=strata,
)

print("Parts:", len(parts))
print("Sample:", criteo.shape)
print("Train :", train_df.shape)
print("Test  :", test_df.shape)
print("Treatment share:", criteo["treatment"].mean())
print("Conversion rate:", criteo["conversion"].mean())


## 2. T-Learner Logistic Regression

In [ ]:

def make_logistic():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42,
        )),
    ])

t_control = make_logistic()
t_treated = make_logistic()

control_train = train_df[train_df["treatment"] == 0]
treated_train = train_df[train_df["treatment"] == 1]

t_control.fit(
    control_train[feature_cols],
    control_train["conversion"],
)
t_treated.fit(
    treated_train[feature_cols],
    treated_train["conversion"],
)

p0_t = t_control.predict_proba(test_df[feature_cols])[:, 1]
p1_t = t_treated.predict_proba(test_df[feature_cols])[:, 1]
uplift_t = p1_t - p0_t


## 3. S-Learner HistGradientBoosting

In [ ]:

s_features = feature_cols + ["treatment"]

s_learner = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.08,
        max_iter=150,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42,
    )),
])

s_learner.fit(
    train_df[s_features],
    train_df["conversion"],
)

X1 = test_df[feature_cols].copy()
X1["treatment"] = 1

X0 = test_df[feature_cols].copy()
X0["treatment"] = 0

p1_s = s_learner.predict_proba(X1[s_features])[:, 1]
p0_s = s_learner.predict_proba(X0[s_features])[:, 1]
uplift_s = p1_s - p0_s


## 4. Uplift evaluation

In [ ]:

def cumulative_incremental_gain(y, treatment, uplift_score):
    order = np.argsort(uplift_score)[::-1]
    y = np.asarray(y)[order]
    t = np.asarray(treatment)[order]

    n_t = np.cumsum(t == 1)
    n_c = np.cumsum(t == 0)

    y_t = np.cumsum(y * (t == 1))
    y_c = np.cumsum(y * (t == 0))

    expected_control = np.divide(
        y_c * n_t,
        n_c,
        out=np.zeros_like(y_c, dtype=float),
        where=n_c > 0,
    )

    gain = y_t - expected_control
    population = np.arange(1, len(y) + 1) / len(y)
    return population, gain

def normalized_qini_area(y, treatment, uplift_score):
    x, gain = cumulative_incremental_gain(y, treatment, uplift_score)

    scale = abs(gain[-1])
    normalized = gain / scale if scale > 0 else gain
    random_line = x * normalized[-1]

    return float(np.trapezoid(normalized - random_line, x))

def uplift_at_fraction(y, treatment, uplift_score, fraction):
    n = max(1, int(np.ceil(len(y) * fraction)))
    idx = np.argsort(uplift_score)[::-1][:n]

    y_sub = np.asarray(y)[idx]
    t_sub = np.asarray(treatment)[idx]

    y_t = y_sub[t_sub == 1]
    y_c = y_sub[t_sub == 0]

    t_rate = y_t.mean() if len(y_t) else np.nan
    c_rate = y_c.mean() if len(y_c) else np.nan

    return t_rate - c_rate

y_test = test_df["conversion"].to_numpy()
t_test = test_df["treatment"].to_numpy()

rows = []
for name, score in [
    ("T-Learner Logistic", uplift_t),
    ("S-Learner HistGB", uplift_s),
]:
    rows.append({
        "model": name,
        "normalized_qini_area": normalized_qini_area(y_test, t_test, score),
        "uplift_at_10pct": uplift_at_fraction(y_test, t_test, score, 0.10),
        "uplift_at_20pct": uplift_at_fraction(y_test, t_test, score, 0.20),
    })

uplift_model_comparison = (
    pd.DataFrame(rows)
    .sort_values("normalized_qini_area", ascending=False)
)

display(uplift_model_comparison)


In [ ]:

fig_qini, ax = plt.subplots(figsize=(9, 5))

for name, score in [
    ("T-Learner Logistic", uplift_t),
    ("S-Learner HistGB", uplift_s),
]:
    x, gain = cumulative_incremental_gain(y_test, t_test, score)
    scale = abs(gain[-1])
    plot_gain = gain / scale if scale > 0 else gain
    ax.plot(x, plot_gain, label=name)

ax.plot([0, 1], [0, 1], linestyle="--", label="Random targeting")
ax.set_title("Criteo Uplift — Cumulative Incremental Gain")
ax.set_xlabel("Fraction of population targeted")
ax.set_ylabel("Normalized cumulative incremental gain")
ax.legend()
fig_qini.tight_layout()
plt.show()


## 5. Uplift by predicted decile

In [ ]:

best_model_name = uplift_model_comparison.iloc[0]["model"]
best_uplift = uplift_s if best_model_name == "S-Learner HistGB" else uplift_t

scored = test_df[["treatment", "conversion"]].copy()
scored["predicted_uplift"] = best_uplift

# Decile 1 = highest predicted uplift.
rank = scored["predicted_uplift"].rank(method="first", ascending=False)
scored["uplift_decile"] = pd.qcut(
    rank,
    10,
    labels=False,
    duplicates="drop",
) + 1

decile_rows = []
for decile, group in scored.groupby("uplift_decile"):
    treated = group.loc[group["treatment"] == 1, "conversion"]
    control = group.loc[group["treatment"] == 0, "conversion"]

    t_rate = treated.mean() if len(treated) else np.nan
    c_rate = control.mean() if len(control) else np.nan

    decile_rows.append({
        "uplift_decile": int(decile),
        "rows": len(group),
        "treatment_conversion_rate": t_rate,
        "control_conversion_rate": c_rate,
        "observed_uplift": t_rate - c_rate,
        "mean_predicted_uplift": group["predicted_uplift"].mean(),
    })

uplift_by_decile = pd.DataFrame(decile_rows).sort_values("uplift_decile")
display(uplift_by_decile)


In [ ]:

fig_deciles, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    uplift_by_decile["uplift_decile"].astype(str),
    100 * uplift_by_decile["observed_uplift"],
)
ax.axhline(0, linestyle="--")
ax.set_title(f"Observed Conversion Uplift by Decile — {best_model_name}")
ax.set_xlabel("Predicted uplift decile (1 = highest)")
ax.set_ylabel("Observed uplift (percentage points)")
fig_deciles.tight_layout()
plt.show()


## 6. Ranked targeting output

In [ ]:

targeting = test_df[feature_cols + ["treatment", "conversion"]].copy()
targeting["predicted_uplift"] = best_uplift
targeting = targeting.sort_values(
    "predicted_uplift", ascending=False
).reset_index(drop=True)

targeting["target_rank"] = np.arange(1, len(targeting) + 1)
targeting["target_percentile"] = targeting["target_rank"] / len(targeting)
targeting["recommended_action"] = np.where(
    targeting["target_percentile"] <= 0.20,
    "TARGET",
    "DO_NOT_PRIORITIZE",
)

top_uplift_targets = targeting.head(100).copy()
display(top_uplift_targets.head(20))


In [ ]:

if best_model_name == "S-Learner HistGB":
    artifact = {
        "model_type": "S-Learner HistGradientBoosting",
        "model": s_learner,
        "features": feature_cols,
        "treatment_feature": "treatment",
    }
else:
    artifact = {
        "model_type": "T-Learner Logistic Regression",
        "control_model": t_control,
        "treatment_model": t_treated,
        "features": feature_cols,
    }

joblib.dump(
    artifact,
    MODELS_DIR / "criteo_uplift_model.joblib",
)
print("Best uplift model:", best_model_name)


## Save compact Notebook 09 results

In [ ]:

RESULTS_DIR = PROJECT_ROOT / "results" / "09_uplift_modeling_and_targeting"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sample_summary = pd.DataFrame([{
    "rows": len(criteo),
    "train_rows": len(train_df),
    "test_rows": len(test_df),
    "treatment_share": criteo["treatment"].mean(),
    "conversion_rate": criteo["conversion"].mean(),
    "sample_per_part": SAMPLE_PER_PART,
    "parts": len(parts),
}])

RESULT_TABLES = {
    "development_sample_summary": sample_summary,
    "uplift_model_comparison": uplift_model_comparison,
    "uplift_by_decile": uplift_by_decile,
    "top_uplift_targets": top_uplift_targets,
}
RESULT_FIGURES = {
    "qini_curves": fig_qini,
    "uplift_by_decile": fig_deciles,
}

for name, table in RESULT_TABLES.items():
    table.to_csv(TABLES_DIR / f"{name}.csv", index=False)
for name, fig in RESULT_FIGURES.items():
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=200, bbox_inches="tight")

print("Tables saved :", len(RESULT_TABLES))
print("Figures saved:", len(RESULT_FIGURES))
print("Results:", RESULTS_DIR)
print("Models:", MODELS_DIR)
